# NV-Diamond Quantum Spin Dynamics — Student Notebook
**Student:** [Name]  
**Period:** [Period]  
**Date:** [Date]  
**Environment:** [Colab / OnDemand JupyterHub / VS Code]

**Instructions:** Read Parts 0–3 in the GitHub repository. Complete **DO NOW** predictions before simulations. Edit `TODO` code cells; do not modify **PROVIDED** utilities. Put independent physics responses in Markdown cells. The completed notebook will be exported to a **PDF**.

## Part 0 — Setup and Analytical Warm-Up
### CODE — PROVIDED · Environment and basis

In [ ]:
# Run this cell first. If QuTiP is missing, install it in a separate cell:
# %pip install qutip numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from matplotlib.figure import Figure
from matplotlib.patches import Patch

zero, one = qt.basis(2, 0), qt.basis(2, 1)
psi_plus = (zero + one).unit()
sx, sy, sz = qt.sigmax(), qt.sigmay(), qt.sigmaz()
projector0, projector1 = zero * zero.dag(), one * one.dag()
figures = {}               # Register Figures 1–4 after creating them.
print(f"QuTiP {qt.__version__} — environment ready")

### CODE — PROVIDED · Shared simulation and plotting utilities (run once)
**Axis overlays:** `plot_bloch(..., omega_z=..., omega_x=..., axis_unit=..., axis_scale=...)` draws the effective Hamiltonian components. These are rotation terms, **not** laboratory field vectors. For Figure 3 use the **same** `axis_unit=omega1` and `axis_scale=0.28` on all four spheres; this makes arrow lengths directly comparable.


In [ ]:
def simulate_spin(H, initial_state, times):
    """Numerical H is H/hbar (rad/us); times are in us.

    ALL time-evolution simulations return the same fields:
    times, states, p0, p1, bloch.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 1 or len(times) < 2 or np.any(np.diff(times) <= 0):
        raise ValueError("times must be a strictly increasing array of at least two values")
    solution = qt.sesolve(H, initial_state, times)
    states = solution.states
    if len(states) != len(times):
        raise RuntimeError("Solver did not return states at every time point")
    return {
        "times": times,
        "states": states,
        "p0": np.asarray(qt.expect(projector0, states), dtype=float),
        "p1": np.asarray(qt.expect(projector1, states), dtype=float),
        "bloch": np.asarray([qt.expect(op, states) for op in (sx, sy, sz)], dtype=float),
    }


def fluorescence(p0, p1, F0=100, F1=70):
    """Expected photons in a fixed readout window."""
    return F0 * np.asarray(p0) + F1 * np.asarray(p1)


def plot_bloch(ax, simulation, title="", color="tab:blue", *,
               omega_z=None, omega_x=None, axis_unit=None, axis_scale=0.90):
    """Draw a Bloch trajectory and its Hamiltonian rotation components.

    omega_z: coefficient of sigma_z (rad/us), labeled omega in Part 1
             or omega_0 in Part 2, depending on the calling cell.
    omega_x: coefficient of sigma_x (rad/us), labeled omega_1.
    axis_unit: common angular-frequency unit used to scale component arrows;
               hold constant across all four detunings for honest comparison.
    axis_scale: Bloch-coordinate arrow length per axis_unit.

    Arrows represent EFFECTIVE ROTATION TERMS in the Hamiltonian, not physical
    laboratory magnetic/microwave field vectors. Zero components are omitted.
    """
    u = np.linspace(0, 2*np.pi, 35)
    v = np.linspace(0, np.pi, 19)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(x, y, z, color="0.83", linewidth=0.35, alpha=0.45)
    for axis, label in [(0, 'x'), (1, 'y'), (2, 'z')]:
        vec = np.zeros(3)
        vec[axis] = 1.16
        ax.plot([-vec[0], vec[0]], [-vec[1], vec[1]],
                [-vec[2], vec[2]], color='0.58', linewidth=0.7)
        ax.text(*vec, label)

    xyz = np.asarray(simulation['bloch'])
    ax.plot(*xyz, color=color, linewidth=2)
    ax.scatter(*xyz[:, 0], color='green', s=38, depthshade=False)
    ax.scatter(*xyz[:, -1], color=color, s=28, depthshade=False)
    direction_at = max(0, min(len(simulation['times'])//8,
                              len(simulation['times']) - 2))
    delta = xyz[:, direction_at + 1] - xyz[:, direction_at]
    if np.linalg.norm(delta) > 1e-8:
        ax.quiver(*xyz[:, direction_at], *delta, color=color,
                  arrow_length_ratio=0.5, length=1, normalize=False)

    # Show the separate z and x terms rather than a combined effective axis.
    # Lengths are proportional to their absolute angular-frequency values.
    if omega_z is not None or omega_x is not None:
        components = [(omega_z, 2, 'tab:orange', r'$\omega$' if omega_x is None
                       else r'$\omega_0$'),
                      (omega_x, 0, 'tab:purple', r'$\omega_1$')]
        active = [abs(float(w)) for w, _, _, _ in components if w is not None and abs(w) > 1e-12]
        unit = float(axis_unit) if axis_unit is not None else (max(active) if active else 1.0)
        if unit <= 0:
            raise ValueError('axis_unit must be positive')
        for value, coordinate, arrow_color, label in components:
            if value is None or np.isclose(value, 0):
                continue
            arrow = np.zeros(3)
            arrow[coordinate] = axis_scale * float(value) / unit
            ax.quiver(0, 0, 0, *arrow, color=arrow_color, linewidth=3,
                      arrow_length_ratio=0.17, normalize=False)
            tip = arrow.copy()
            tip[coordinate] += 0.10 * np.sign(arrow[coordinate])
            ax.text(*tip, label, color=arrow_color, fontsize=11, weight='bold')

    ax.set(xlim=(-1.47, 1.47), ylim=(-1.47, 1.47), zlim=(-1.47, 1.47))
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=22, azim=42)
    ax.set_axis_off()
    # Fixed-position color key keeps the short component readable in print.
    if omega_z is not None:
        z_label = r'$\omega$ (z)' if omega_x is None else r'$\omega_0$ (z)'
        if np.isclose(omega_z, 0):
            z_label += ' = 0'
        ax.text2D(0.02, 0.07, z_label, transform=ax.transAxes,
                  color='tab:orange', fontsize=9, weight='bold')
    if omega_x is not None:
        ax.text2D(0.02, 0.015, r'$\omega_1$ (x)', transform=ax.transAxes,
                  color='tab:purple', fontsize=9, weight='bold')
    ax.set_title(title, fontsize=10)
    return ax


def check_normalization(sim):
    """Return maximum state-norm and population-sum errors."""
    norm_error = max(abs(state.norm() - 1) for state in sim['states'])
    pop_error = np.max(np.abs(sim['p0'] + sim['p1'] - 1))
    return float(norm_error), float(pop_error)

print("Shared simulation and plotting functions ready.")

### DO NOW — Analytical Warm-Up · Work independently
Initial state: $|\psi(0)\rangle=(|0\rangle+|1\rangle)/\sqrt2$; $\hat H=\hbar\omega\sigma_z/2$.

**Energy eigenvalues:** [Your calculation]

**Time-dependent state:** [Your calculation]

**Populations $P_0(t)$ and $P_1(t)$:** [Your calculation]

**Relative phase and predicted Bloch trajectory:** [Your response]

### CODE — TODO · Verify warm-up with QuTiP

In [ ]:
# TODO: Construct H/hbar for a chosen angular frequency.
# Use simulate_spin(H, psi_plus, times) to verify your calculations.
# Report populations and a sample evolved state below.
omega_warm = 2*np.pi
times_warm = np.linspace(0, 1, 101)
H_warm = None                 # TODO
warmup = None                 # TODO: standard simulation-result dictionary
# print(...)                  # TODO: numerical comparison

**RESULT — Warm-up numerical agreement:** [Brief comparison with your analytical prediction]

## Part 1 — Larmor Precession and Quantum Phase
### DO NOW — Predictions (before running code)
**1. Initial Bloch location:** [Answer]  
**2. Trajectory:** [Answer]  
**3. Behavior of $P_0,P_1$:** [Answer]  
**4. Effect of doubling $\omega$:** [Answer]

### CODE — TODO · Larmor simulation (Figure 1)

In [ ]:
omega = 2*np.pi                         # rad/us
times1 = np.linspace(0, 2, 501)         # us
H1 = None                               # TODO: H/hbar = omega*sz/2
part1 = None                            # TODO: simulate_spin(H1, psi_plus, times1)

### CODE — TODO · Figure 1: Bloch trajectory + populations
**Rotation axis overlay:** Label the $z$-axis term $\omega$ on Figure 1. The arrow represents the effective rotation term, not the physical magnetic field.


In [ ]:
# TODO: Create Figure 1: Bloch sphere + P0/P1 versus time.
# On the sphere, draw the Part 1 z rotation term as a labeled arrow:
# plot_bloch(ax, part1, ..., omega_z=omega, axis_unit=omega)
# Mark initial state and evolution direction (the helper handles these).
# Register the finished figure: figures['Figure 1'] = fig1


**FIGURE 1 CAPTION —** [1–2 sentences connecting precession and unchanged populations]

### CODE — TODO · Double precession frequency and report numerical verification

In [ ]:
omega_fast = 4*np.pi
H1_fast = None                        # TODO: use sz
part1_fast = None                    # TODO: use simulate_spin()

# TODO: Print a compact table comparing for both simulations:
# predicted period 2*pi/abs(omega), number of revolutions in 2 us,
# largest change in P0 and P1, and normalization errors.

### DO NOW — Independent Physics Analysis
**Q1 — Time Evolution Without Population Transfer.** Why does the Bloch vector move while $P_0,P_1$ remain constant? Distinguish relative phase from basis-state probabilities.

[Type your answer here]

**Q2 — Magnetic Fields and Precession.** Why does an energy difference change relative-phase accumulation, and how does that connect to the NV spin and a magnetic field?

[Type your answer here]

## Part 2A — Resonant Microwave Driving
### DO NOW — Predictions (before running code)
**1. Initial Bloch location and rotation axis:** [Answer]  
**2. Behavior of $P_0,P_1$:** [Answer]  
**3. First $\pi$-pulse duration:** [Answer]  
**4. Effect of doubling $\omega_1$:** [Answer]

### CODE — TODO · Define NV Hamiltonian ONCE for Parts 2 and 3

In [ ]:
def nv_hamiltonian(omega0, omega1):
    """Return effective H/hbar in rad/us.
    omega0 = omega_NV - omega_drive (signed detuning)
    omega1 = microwave coupling strength
    """
    # TODO: Return (omega0*sz + omega1*sx)/2 using QuTiP operators.
    raise NotImplementedError("Implement nv_hamiltonian once; reuse it in Parts 2–3")

### CODE — TODO · Resonant Rabi simulation

In [ ]:
omega1 = 2*np.pi
omega0 = 0
times2 = np.linspace(0, 2, 1001)        # finer spacing resolves strong detuning
H_res = nv_hamiltonian(omega0, omega1)
part2_res = None                          # TODO: simulate_spin(..., zero, times2)
# TODO: Find the first numerical pi-pulse near t = pi/omega1.

### CODE — TODO · Figure 2: Resonant Bloch sphere + populations
**Rotation axis overlay:** Show the $x$-axis microwave coupling $\omega_1$ on the resonant Bloch sphere.


In [ ]:
# TODO: Create Figure 2: Bloch sphere + resonant population curves.
# Show the microwave x rotation term (omega1) as a labeled arrow:
# plot_bloch(ax, part2_res, ..., omega_x=omega1, axis_unit=omega1)
# Mark the first pi-pulse on the population plot.
# Register the finished figure: figures['Figure 2'] = fig2


**FIGURE 2 CAPTION —** [Explain how the trajectory corresponds to population transfer]

### CODE — TODO · Double coupling and verify π-pulse duration

In [ ]:
omega1_fast = 4*np.pi
H_res_fast = nv_hamiltonian(0, omega1_fast)
part2_fast = None                # TODO: simulate_spin(H_res_fast, zero, times2)
# TODO: Print a two-row comparison of predicted and numerical first pi-pulse
# durations for omega1 and omega1_fast. Compare population periods.

## Part 2B — Approaching Resonance
### DO NOW — Predictions
With $\omega_1=2\pi$ rad/µs fixed, approach resonance through $\omega_0/\omega_1=4,2,1,0$.

**1. Closest to north pole / reaches south pole:** [Answer]  
**2. Trend in max $P_1$:** [Answer]  
**3. Trend in effective rotation axis:** [Answer]  
**4. Trend in oscillation speed:** [Answer]

### CODE — TODO · Four detunings, one solver and one dictionary

In [ ]:
detuning_ratios = [4, 2, 1, 0]
detuning_results = {}

for ratio in detuning_ratios:
    omega0_case = ratio * omega1
    # TODO: Build H with nv_hamiltonian(omega0_case, omega1).
    # TODO: Store simulate_spin(H, zero, times2) at detuning_results[ratio].
    pass

### CODE — TODO · Figure 3: Four Bloch spheres + four population curves
**Rotation axis overlays:** Show $\omega_0$ along $z$ and $\omega_1$ along $x$, using proportional lengths and the same scale on all four spheres. Do not draw a resultant arrow.


In [ ]:
# TODO: Create Figure 3: four Bloch spheres (ratios 4,2,1,0),
# plus one shared P1(t) plot below the spheres.
# On EACH sphere show both Hamiltonian terms:
#   plot_bloch(ax, detuning_results[ratio], ...,
#              omega_z=ratio*omega1, omega_x=omega1,
#              axis_unit=omega1, axis_scale=0.28)
# Use the SAME axis_unit, axis_scale, view angle, and colors on all four.
# The x-component has equal length in all four; z-components have lengths
# proportional to 4:2:1:0. At resonance omega0=0, the z arrow is omitted.
# Label curves and indicate direction of time evolution.
# Register the finished figure: figures['Figure 3'] = fig3


**FIGURE 3 CAPTION —** [Connect detuning, axis tilt, transfer amplitude, and generalized Rabi frequency]

### CODE — TODO · Detuning comparison table and checks

In [ ]:
# TODO: Print 4 rows: ratio, simulated max P1, theoretical max P1,
# and Omega = sqrt(omega0**2 + omega1**2) (rad/us).
# Verify expected max P1 values: 1/17, 1/5, 1/2, 1.
# Use max(result['p1']) for the simulated values.

### DO NOW — Independent Physics Analysis
**Q3 — Resonant Spin Control.** Compared with Part 1, how does the resonant Hamiltonian change the rotation axis and allow a complete spin flip with a $\pi$-pulse?

[Type your answer here]

**Q4 — Approaching Resonance and Magnetometry.** Use your four Bloch trajectories and population curves to explain how decreasing detuning changes the rotation axis, maximum transfer, and oscillation frequency. How could a changing magnetic field alter detuning and inform magnetometry?

[Type your answer here]

## Part 3 — Optical Readout and NV Magnetometry
### DO NOW — Predict the fluorescence spectrum
For $\tau=0.5$ µs, $F_0=100$, $F_1=70$:

**1. Far from resonance, fluorescence near $F_0$ or $F_1$?** [Answer]  
**2. Trend toward resonance:** [Answer]  
**3. Location of minimum fluorescence:** [Answer]

### CODE — TODO · Frequency sweep using the SAME simulation result format

In [ ]:
tau = np.pi / omega1                  # 0.5 us, fixed for EVERY frequency
# Odd number of samples ensures both original and shifted resonances lie on grid.
drive_offsets = np.linspace(-4*omega1, 4*omega1, 201)


def sweep_fluorescence(drive_offsets, nv_shift=0.0):
    """Sweep fixed-duration pulse; reuse nv_hamiltonian and simulate_spin.

    drive_offsets: omega_drive - ORIGINAL omega_NV (rad/us)
    nv_shift: shift of NV transition angular frequency (rad/us)

    Returns original frequency axis, final populations, fluorescence, plus
    each point's standard simulate_spin() result in 'simulations'.
    """
    # TODO: For each offset:
    #   omega0 = nv_shift - offset
    #   H = nv_hamiltonian(omega0, omega1)
    #   sim = simulate_spin(H, zero, [0, tau])
    #   collect sim['p0'][-1] and sim['p1'][-1]
    #   preserve individual standard sim dicts in 'simulations'
    #   convert populations using fluorescence(...)
    # TODO: Return a dictionary containing:
    #   'drive_offsets', 'simulations', 'p0', 'p1', 'fluorescence'
    raise NotImplementedError("Implement sweep by reusing existing functions")

part3_base = None          # TODO: sweep_fluorescence(drive_offsets, nv_shift=0)

### DO NOW — Predict the magnetic-field-induced shift
The transition shifts by $\delta\omega_{\mathrm{NV}}=\omega_1$.

**1. Effect on resonance frequency:** [Answer]  
**2. Direction of fluorescence minimum:** [Answer]  
**3. Effect on minimum fluorescence:** [Answer]

### CODE — TODO · Shift transition, do NOT shift the drive-frequency array

In [ ]:
nv_shift = omega1
# TODO: Reuse the SAME drive_offsets. Only change the transition shift.
part3_shift = None          # TODO: sweep_fluorescence(drive_offsets, nv_shift)

### CODE — TODO · Figure 4: Original and shifted pulsed-ODMR spectra

In [ ]:
# TODO: Draw two fluorescence spectra versus the SAME drive_offsets/(2*pi)
# axis in MHz relative to the original NV transition.
# Mark their central resonance locations (0 MHz and +1 MHz).
# Label expected red photons per readout window.
# Register Figure 4:
# figures['Figure 4'] = fig4

**FIGURE 4 CAPTION —** [Explain how changing the magnetic field shifts the optical resonance]

### CODE — TODO · Numerical resonance report

In [ ]:
# TODO: Use the central minimum of each spectrum to report:
# - Expected transition shift (MHz)
# - Detected central fluorescence minimum (MHz, relative to original resonance)
# - Minimum photon counts
# - Measured shift in MHz
# - Check F(+detuning) = F(-detuning) for unshifted spectrum.
# Note: frequency in MHz = (angular frequency in rad/us) / (2*pi).

### DO NOW — Step 7: Interpret the ODMR resonance shift
**Conceptual question:** Using the measured shift in the NV resonance frequency, what physical quantity could you calculate? Explain the relationship between this quantity and the NV spin's energy-level splitting. **No additional coding or numerical calculation is required.**

**Your answer:** [Type your response here]


### DO NOW — Independent Physics Analysis
**Q5 — From Spin Dynamics to Fluorescence.** Explain why fluorescence decreases near resonance by connecting detuning, spin-flip probability, and spin-dependent fluorescence yields.

[Type your answer here]

**Q6 — Using NV Centers as Magnetometers.** Explain the chain from magnetic-field change to NV energy splitting to resonance-frequency shift and fluorescence measurement. Why measure the resonance rather than fluorescence at an arbitrary drive frequency?

[Type your answer here]

## AI Development Record — One entry per computational part
*If no AI was used, describe independent development and verification. Do not include full transcripts.*

### AI Development Record 1 — Larmor Precession
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

### AI Development Record 2 — Microwave Driving and Rabi Oscillations
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

### AI Development Record 3 — Optical Readout and Magnetometry
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

### Final AI Reflection — Approximately 100 words
How did AI assistance affect your workflow? Identify one error, limitation, or unexpected result you investigated. Explain one physics insight you gained by interpreting the results rather than merely running code. (If no AI was used, describe independent coding and verification.)

[Type approximately 100 words here]

## Final Submission Check
### CODE — PROVIDED · Run after completing the notebook

In [ ]:
# Incomplete tasks are NOT RUN; completed incorrect results are FAIL.
# This checks numerical output and figure objects, not written work or plot quality.

checks = []

def add_check(label, test, available=True):
    status = 'NOT RUN' if not available else ('PASS' if bool(test) else 'FAIL')
    checks.append((label, status))
    print(f'{status:8} | {label}')

def valid_sim(v):
    return (isinstance(v, dict)
            and all(k in v for k in ('times','states','p0','p1','bloch'))
            and len(v['states']) >= 2)

def valid_sweep(v):
    return (isinstance(v, dict)
            and all(k in v for k in ('drive_offsets','simulations','p0','p1','fluorescence'))
            and len(v['drive_offsets']) == len(v['p1']) == len(v['fluorescence'])
            and len(v['drive_offsets']) >= 100
            and all(valid_sim(sim) for sim in v['simulations'])
            and len(v['simulations']) == len(v['drive_offsets']))

for i in range(1, 5):
    fig = figures.get(f'Figure {i}')
    add_check(f'Figure {i} registered', isinstance(fig, Figure), fig is not None)

if valid_sim(globals().get('part1')):
    norm_err, pop_err = check_normalization(part1)
    add_check('Part 1 state and population normalization', max(norm_err,pop_err) < 1e-5)
    add_check('Part 1 populations constant',
              max(np.ptp(part1['p0']), np.ptp(part1['p1'])) < 1e-5)
else:
    add_check('Part 1 state and population normalization', False, False)
    add_check('Part 1 populations constant', False, False)

if valid_sim(globals().get('part1_fast')):
    add_check('Part 1 doubled-frequency normalization', max(check_normalization(part1_fast)) < 1e-5)
else:
    add_check('Part 1 doubled-frequency normalization', False, False)

if valid_sim(globals().get('part2_res')):
    norm_err, pop_err = check_normalization(part2_res)
    add_check('Resonant Rabi normalization', max(norm_err,pop_err) < 1e-5)
    i_pi = int(np.argmin(np.abs(part2_res['times'] - np.pi/omega1)))
    add_check('Resonant pi-pulse spin flip', part2_res['p1'][i_pi] > 0.995)
else:
    add_check('Resonant Rabi normalization', False, False)
    add_check('Resonant pi-pulse spin flip', False, False)

if valid_sim(globals().get('part2_fast')):
    i_fast = int(np.argmin(np.abs(part2_fast['times'] - np.pi/(2*omega1))))
    add_check('Doubled-coupling pi pulse', part2_fast['p1'][i_fast] > 0.995)
else:
    add_check('Doubled-coupling pi pulse', False, False)

cases = globals().get('detuning_results')
if isinstance(cases, dict) and all(valid_sim(cases.get(r)) for r in [4,2,1,0]):
    add_check('Four detuning simulations normalized',
              all(max(check_normalization(cases[r])) < 1e-5 for r in [4,2,1,0]))
    add_check('Detuning max transfer agrees with theory',
              all(abs(np.max(cases[r]['p1']) - 1/(1+r*r)) < 0.012 for r in [4,2,1,0]))
else:
    add_check('Four detuning simulations normalized', False, False)
    add_check('Detuning max transfer agrees with theory', False, False)

base, shift = globals().get('part3_base'), globals().get('part3_shift')
if valid_sweep(base) and valid_sweep(shift):
    norm_ok = all(max(check_normalization(sim)) < 1e-5
                  for dat in (base,shift) for sim in dat['simulations'])
    add_check('Frequency-sweep populations normalized', norm_ok)
    all_f = np.concatenate([np.asarray(base['fluorescence']), np.asarray(shift['fluorescence'])])
    add_check('Fluorescence within [70, 100]', np.min(all_f)>=70-1e-5 and np.max(all_f)<=100+1e-5)
    same_axis = np.allclose(base['drive_offsets'], shift['drive_offsets'])
    add_check('Both ODMR spectra share drive-frequency axis', same_axis)
    add_check('Unshifted ODMR response symmetric',
              np.allclose(np.asarray(base['fluorescence']), np.asarray(base['fluorescence'])[::-1], atol=0.02))
    # These are central minima because a resonant pi-pulse yields the global fluorescence minimum.
    f0_MHz = base['drive_offsets'][np.argmin(base['fluorescence'])]/(2*np.pi)
    f1_MHz = shift['drive_offsets'][np.argmin(shift['fluorescence'])]/(2*np.pi)
    add_check('Original and shifted central resonances',
              abs(f0_MHz)<0.03 and abs(f1_MHz-1)<0.03)
    add_check('Magnetic-field-induced shift = +1 MHz', abs((f1_MHz-f0_MHz)-1)<0.03)
else:
    for label in ('Frequency-sweep populations normalized','Fluorescence within [70, 100]',
                  'Both ODMR spectra share drive-frequency axis','Unshifted ODMR response symmetric',
                  'Original and shifted central resonances','Magnetic-field-induced shift = +1 MHz'):
        add_check(label, False, False)

from collections import Counter
counts = Counter(status for _,status in checks)
print(f"\nSUMMARY: {counts['PASS']} PASS | {counts['FAIL']} FAIL | {counts['NOT RUN']} NOT RUN")
print('Review captions, Q1–Q6, AI records, reflection, and PDF manually; these cannot be verified here.')

### FINAL DO NOW — PDF submission checklist
- [ ] Figures 1–4 are visible, correctly labeled, and captioned.
- [ ] Predictions and numerical reports are completed.
- [ ] Q1–Q6 were independently answered.
- [ ] AI records 1–3 and final reflection are completed.
- [ ] Submission Check was run; errors investigated.
- [ ] Notebook was executed top-to-bottom and saved.
- [ ] Exported PDF was opened and checked for missing or cut-off content.

**Submit one PDF named `NV_Spin_Dynamics_Lastname.pdf` to the LMS. Keep your `.ipynb`.**